
# Semantic Memory - Exploratory Data Analysis

**Paper context:** *The Role of System 1 and System 2 Semantic Memory Structure in Human and LLM Biases* (Abramski, Rossetti & Stella, 2026)

## 1. Introduction

This notebook explores the datasets used in the paper, focusing on semantic associations produced by humans and large language models (LLMs). The main goal is to understand how the data is structured, identify common patterns, and compare human and LLM-generated word associations.

We will use descriptive statistics and visualizations to explore the data, rather than reproduce the full experiments from the paper.



### 1.1 Environment Setup

We will use Python and Pandas to load, inspect, and analyze the datasets. NumPy will be used for numerical operations, while Plotly will help us create interactive visualizations.


In [3]:

import pandas as pd
import numpy as np
import plotly.express as px

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)



## 2. Dataset Description

This analysis focuses on two sources of semantic word associations:

- **SWOW (Small World of Words):** A dataset containing word associations collected from human participants. Given a cue word, participants provide words that come to mind.

- **LWOW (Large World of Words):** A dataset of word associations generated by large language models. It allows us to explore how LLM-generated associations compare with human responses.

We will mainly focus on these two datasets to investigate differences and similarities in word association patterns.

The exact dataset structure, available columns, and number of observations will be examined after loading the data.



## 3. Data Loading & Initial Inspection

In this section, we load the SWOW and LWOW datasets and examine their structure.

We will check the available columns, data types, number of observations, and missing values before starting the analysis.


In [1]:
from pathlib import Path

DATA_DIR = Path("data")

print("Available datasets:")

for path in sorted(DATA_DIR.rglob("*.csv")):
    print(path)

Available datasets:
data/lwow/FA_Llama3.csv
data/lwow/FA_Mistral.csv
data/swow/SWOW-EN.R100.20180827.csv



### 3.1 Loading the SWOW Dataset

We will start by examining the SWOW dataset, which contains word associations collected from human participants.

First, we will load a small sample to check the file structure and make sure the data can be read correctly.


In [4]:

swow_path = DATA_DIR / "swow" / "SWOW-EN.R100.20180827.csv"

swow_preview = pd.read_csv(swow_path, nrows=5)

display(swow_preview)


,Unnamed: 0,id,participantID,age,gender,nativeLanguage,country,education,created_at,cue,R1,R2,R3
0,1,29,3,33,Fe,United States,Australia,NaN,2011-08-12 02:19:38,although,nevertheless,yet,but
1,2,30,3,33,Fe,United States,Australia,NaN,2011-08-12 02:19:38,deal,no,cards,shake
2,3,31,3,33,Fe,United States,Australia,NaN,2011-08-12 02:19:38,music,notes,band,rhythm
3,4,32,3,33,Fe,United States,Australia,NaN,2011-08-12 02:19:38,inform,tell,rat on,NaN
4,5,33,3,33,Fe,United States,Australia,NaN,2011-08-12 02:19:38,way,path,via,method


In [5]:
# now we can load the full dataset

swow_path = DATA_DIR / "swow" / "SWOW-EN.R100.20180827.csv"

swow = pd.read_csv(swow_path, low_memory=False)

display(swow.head())


,Unnamed: 0,id,participantID,age,gender,nativeLanguage,country,education,created_at,cue,R1,R2,R3
0,1,29,3,33,Fe,United States,Australia,NaN,2011-08-12 02:19:38,although,nevertheless,yet,but
1,2,30,3,33,Fe,United States,Australia,NaN,2011-08-12 02:19:38,deal,no,cards,shake
2,3,31,3,33,Fe,United States,Australia,NaN,2011-08-12 02:19:38,music,notes,band,rhythm
3,4,32,3,33,Fe,United States,Australia,NaN,2011-08-12 02:19:38,inform,tell,rat on,NaN
4,5,33,3,33,Fe,United States,Australia,NaN,2011-08-12 02:19:38,way,path,via,method


In [6]:

print(f"Number of rows: {swow.shape[0]:,}")
print(f"Number of columns: {swow.shape[1]}")
print(f"Columns: {swow.columns.tolist()}")


Number of rows: 1,228,200
Number of columns: 13
Columns: ['Unnamed: 0', 'id', 'participantID', 'age', 'gender', 'nativeLanguage', 'country', 'education', 'created_at', 'cue', 'R1', 'R2', 'R3']



### 3.2 Data Types & Missing Values

Before preprocessing the data, we need to check the data types and identify missing values.

This will help us understand which columns might require cleaning and whether missing responses could affect our analysis.


In [7]:

swow.info()

<class 'pandas.DataFrame'>
RangeIndex: 1228200 entries, 0 to 1228199
Data columns (total 13 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   Unnamed: 0      1228200 non-null  int64  
 1   id              1228200 non-null  int64  
 2   participantID   1228200 non-null  int64  
 3   age             1228200 non-null  int64  
 4   gender          1228200 non-null  str    
 5   nativeLanguage  1228200 non-null  str    
 6   country         1223518 non-null  str    
 7   education       536092 non-null   float64
 8   created_at      1228200 non-null  str    
 9   cue             1228100 non-null  str    
 10  R1              1197104 non-null  str    
 11  R2              1148452 non-null  str    
 12  R3              1058117 non-null  str    
dtypes: float64(1), int64(4), str(8)
memory usage: 121.8 MB


In [8]:

missing_values = pd.DataFrame({
    "Missing Count": swow.isna().sum(),
    "Missing (%)": swow.isna().mean() * 100
})

display(missing_values.round(2))


,Missing Count,Missing (%)
Unnamed: 0,0,0.00
id,0,0.00
participantID,0,0.00
age,0,0.00
gender,0,0.00
nativeLanguage,0,0.00
country,4682,0.38
education,692108,56.35
created_at,0,0.00
cue,100,0.01



**Initial observations:**

- Most columns have very few missing values.
- The `education` column has a high percentage of missing values (56.35%), but it is not relevant to our main semantic association analysis.
- Missing responses increase from R1 (2.53%) to R3 (13.85%). This could indicate that some participants did not provide all three responses.
- Only 0.01% of observations have missing cue words.

**Preprocessing considerations:**

We should avoid removing entire rows just because some values are missing. For example, an observation with valid R1 and R2 responses is still useful even if R3 is missing.

We will investigate these cases further during data preparation and decide how to handle missing values without losing useful information.



### 3.3 Loading the LWOW Datasets

We will now examine the word associations generated by Llama3 and Mistral.

First, we will load both datasets and inspect their structure. This will help us understand how LLM-generated associations are stored and whether they can be compared with the human responses from SWOW.


In [ ]:

llama_path = DATA_DIR / "lwow" / "FA_Llama3.csv"
mistral_path = DATA_DIR / "lwow" / "FA_Mistral.csv"

llama = pd.read_csv(llama_path, low_memory=False)
mistral = pd.read_csv(mistral_path, low_memory=False)

print("Llama3:")
display(llama.head())

# edit: why does mistal return NaN
print("Mistral:")
display(mistral.head())


Llama3:


,cue,R1,R2,R3
0,a,absorbent,acoustic,accelerate
1,a,absorbent,acoustic,accelerate
2,a,absorbent,acoustic,accelerate
3,a,abundance,accentuate,affect
4,a,acorn,airplane,astronaut


Mistral:


,cue,R1,R2,R3
0,a,NaN,NaN,NaN
1,a,NaN,NaN,NaN
2,a,NaN,NaN,NaN
3,a,NaN,NaN,NaN
4,a,NaN,NaN,NaN


In [10]:

for name, df in [("Llama3", llama), ("Mistral", mistral)]:
    print(f"\n{name}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print(f"Column names: {df.columns.tolist()}")



Llama3
Rows: 1,154,500
Columns: 4
Column names: ['cue', 'R1', 'R2', 'R3']

Mistral
Rows: 1,154,500
Columns: 4
Column names: ['cue', 'R1', 'R2', 'R3']


In [ ]:

# show the first rows containing at least one valid response
display(
    mistral.dropna(subset=["R1", "R2", "R3"], how="all").head(10)
)


,cue,R1,R2,R3
13,a,airplane,apple,ant
14,a,also,especially,furthermore
15,a,an,and,at
16,a,an,apple,afternoon
17,a,an,the,NaN
18,a,and,NaN,NaN
19,a,and,about,NaN
20,a,and,also,association
21,a,and,aspect,assume
22,a,and,but,NaN


In [12]:

for name, df in [("Llama3", llama), ("Mistral", mistral)]:
    print(f"\n{name}")

    print("Missing values (%):")
    print((df[["R1", "R2", "R3"]].isna().mean() * 100).round(2))

    all_missing = df[["R1", "R2", "R3"]].isna().all(axis=1).sum()

    print(f"Rows with all responses missing: {all_missing:,}")
    print(f"Percentage: {all_missing / len(df) * 100:.2f}%")



Llama3
Missing values (%):
R1    1.90
R2    1.93
R3    6.18
dtype: float64
Rows with all responses missing: 21,917
Percentage: 1.90%

Mistral
Missing values (%):
R1    4.07
R2    4.87
R3    7.98
dtype: float64
Rows with all responses missing: 46,969
Percentage: 4.07%



**Initial observations:**

- Both LWOW datasets contain 1,154,500 rows and share the same four columns: `cue`, `R1`, `R2`, and `R3`.
- Mistral has more missing responses than Llama3 across all three response positions (this is why we see NaN when returning `head()`)
- R3 has the highest percentage of missing values in both datasets.
- Around 1.90% of Llama3 rows and 4.07% of Mistral rows have no recorded responses.
- A small inspection of the original Mistral CSV confirmed that some response fields are genuinely empty. These missing values are not simply caused by Pandas interpreting text as NaN (see below)


**Preprocessing considerations:**

Both datasets appear suitable for further analysis. However, differences in missing responses could affect comparisons of association frequency and lexical diversity.

We should account for the number of valid responses when comparing models and investigate how missing values are represented in the original files.


In [14]:

# small checkup, inspect the raw values from the beginning of the Mistral CSV
mistral_raw = pd.read_csv(
    mistral_path,
    nrows=25,
    keep_default_na=False
)

display(mistral_raw.head(25))


,cue,R1,R2,R3
0,a,,,
1,a,,,
2,a,,,
3,a,,,
4,a,,,
5,a,,,
6,a,,,
7,a,,,
8,a,,,
9,a,,,
